# Servidor DeepSeek-V4-Pro-Qwen3.5-9B (heretic) para opencode

Correr en un **notebook Kaggle separado** con:
- Settings → **Accelerator: GPU T4 x2**
- Settings → **Internet: ON**

Deja el modelo servido como API OpenAI-compatible en el puerto 8000 y lo expone
con un túnel público de Cloudflare. La URL que imprime la última celda es la que
va en `baseURL` del provider `kaggle-deepseek` de opencode
(archivo `/kaggle/working/opencode_cloud/config/opencode.json` en la workstation).

⚠️ La URL cambia cada vez que reinicies este notebook: actualizar `baseURL` y
reiniciar opencode.

In [ ]:
!pip install -q -U vllm   # ~10 min

In [ ]:
import subprocess, time

MODEL = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"

comando = [
    "vllm", "serve", MODEL,
    "--tensor-parallel-size", "2",   # reparte entre las 2 T4
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.92",
    "--port", "8000",
]
log = open("/kaggle/working/vllm.log", "w")
subprocess.Popen(comando, stdout=log, stderr=subprocess.STDOUT)
print("Servidor arrancando (descarga ~18 GB): mira /kaggle/working/vllm.log")

# Espera activa hasta que la API responda
import urllib.request
for i in range(60):            # hasta 30 min
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print("API lista tras", (i+1)*30, "segundos")
        break
    except Exception:
        print(f"[{(i+1)*30}s] aún cargando...")
else:
    print("Timeout. Últimas líneas del log:")
    print("\n".join(open("/kaggle/working/vllm.log").readlines()[-10:]))

In [ ]:
# Túnel público (gratuito, sin cuenta). La URL impresa es la que va en opencode.
import subprocess, time, re

if not __import__("os").path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

log = open("/kaggle/working/tunnel.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                 stdout=log, stderr=subprocess.STDOUT)

time.sleep(15)
contenido = open("/kaggle/working/tunnel.log").read()
m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", contenido)
print("URL del túnel para opencode:", m.group(0) if m else "(aún no aparece, revisa /kaggle/working/tunnel.log)")

In [ ]:
# Prueba rápida del endpoint (debe responder en JSON)
import urllib.request, json
req = urllib.request.Request(
    "http://localhost:8000/v1/chat/completions",
    data=json.dumps({
        "model": "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic",
        "messages": [{"role": "user", "content": "Decime hola en una frase."}],
        "max_tokens": 100,
    }).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.loads(urllib.request.urlopen(req, timeout=300).read())["choices"][0]["message"]["content"])